### Tools

Models can requests to call tools that perform tasks such as fetching data from a database, searching the web, or running code. Tools are pairing of:

1. A schema including the name of the tool, a description of what it does, and/or argument definations (often a JSON schema).
2. A function that implements the tool's functionality.

In [4]:
import os
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
load_dotenv()

os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY")

model = ChatGoogleGenerativeAI(model="gemini-3.5-flash-lite")
response = model.stream("write a poem about the beauty of nature in 100 words")
for chunk in response:
    print(chunk.text, end="", flush=True)


Gold spills across the waking, velvet hills,
As morning drinks the crystal mountain rills.
An ancient oak extends a leafy hand,
To bless the quiet, deeply breathing land.

A painted bunting flashes through the green,
The brightest notes within a peaceful scene.
The river hums a silver, winding tune,
Beneath the crescent of a fading moon.

Wild orchids bloom where shadows softly play,
And chase the heavy, weary ghosts away.
In every blade of grass, a sacred art,
Nature sings her anthem to the heart.
The wild earth heals, untamed and purely free,
A masterpiece of grace for eyes to see.

In [10]:
### to create a tool
### we import tools from langchain

from langchain.tools import tool

# we use this tool as a decorator to create a tool that can be used by the model to call a function that we define.
@tool
def get_weather(location: str) -> str:
    """Get the weather at a location"""
    return f"The weather at {location} is sunny and 20 degrees."


#bind the tool to the model
model_with_tools = model.bind_tools([get_weather])

In [12]:
response = model_with_tools.invoke("What is the weather in New York?")
print(response)
for tool_call in response.tool_calls:
    print(f"Tool called: {tool_call['name']}")
    print(f"Tool args: {tool_call['args']}")

content=[] additional_kwargs={'function_call': {'name': 'get_weather', 'arguments': '{"location": "New York"}'}, '__gemini_function_call_thought_signatures__': {'call_4607762418928700836': 'EmAKXgFpFH0TH3oH+mtuhw4f1/UztNmJHsyxBxXR2Mpez+YGM9zFr5HXb4VWnGU5FkuPJ6CQVqBNfV7yR41QWaAXH0dYdt5mfbVPNlb4wVJB6U4W+V9I9O5YX3QnhCQV324='}} response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'} id='lc_run--01a1da42-8112-74c4-b73d-b7c35c4319fa-0' tool_calls=[{'name': 'get_weather', 'args': {'location': 'New York'}, 'id': 'call_4607762418928700836', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 52, 'output_tokens': 17, 'total_tokens': 69, 'input_token_details': {'cache_read': 0}}
Tool called: get_weather
Tool args: {'location': 'New York'}


### Tool Execution loop

When a model calls a tool, the tool is executed and the result is returned to the model. The model can then use the result to generate a response to the user.

In [ ]:
#step 1: Model generats tool calls
messages = [{"role": "user", "content": "What is the weather in New York?"}]
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)

#step 2: Execute tools and collect results
for tool_call in ai_msg.tool_calls:
    #Execute the tool with generated arguments
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)
    
#step 3: Pass results back to model for final response
final_response = model_with_tools.invoke(messages)
print(final_response.text)
# The current weather in New York is sunny and 20 degrees.

[{'role': 'user', 'content': 'What is the weather in New York?'}, AIMessage(content=[], additional_kwargs={'function_call': {'name': 'get_weather', 'arguments': '{"location": "New York"}'}, '__gemini_function_call_thought_signatures__': {'call_7794068873607245204': 'EmAKXgFpFH0Tl5eh3VIKShm+5BGLqSWOcx476XOyCTiRdhe9LXlVQjAx/5sGPBsm846a9Y5oAm9LdTIvuXOoZyYQrc5np9W4G921ouWuCGkgHXZFGoO7ZwYqXsfPhbG5fww='}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a1da56-c97a-7650-9c66-9c2f3ad50084-0', tool_calls=[{'name': 'get_weather', 'args': {'location': 'New York'}, 'id': 'call_7794068873607245204', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 52, 'output_tokens': 17, 'total_tokens': 69, 'input_token_details': {'cache_read': 0}})]
The weather in New York is sunny and 20 degrees.


In [15]:
print(messages)

[{'role': 'user', 'content': 'What is the weather in New York?'}, AIMessage(content=[], additional_kwargs={'function_call': {'name': 'get_weather', 'arguments': '{"location": "New York"}'}, '__gemini_function_call_thought_signatures__': {'call_99750276080216963': 'EmAKXgFpFH0TcJQ1leZ0l0Y4BjAk9Im+mj2M/wH6dSwawqtk9gBj1zG12JYh1gVHg7FRPnAEE+9F2ljNBfMXBkxhPW4K5R+NRYsr7xAM+5eJm82VPOUvTj1cm0NimAfVp1g='}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a1da56-03ee-760c-b00e-3b7517826fbd-0', tool_calls=[{'name': 'get_weather', 'args': {'location': 'New York'}, 'id': 'call_99750276080216963', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 52, 'output_tokens': 17, 'total_tokens': 69, 'input_token_details': {'cache_read': 0}}), ToolMessage(content='The weather at New York is sunny and 20 degrees.', name='get_weather', tool_call_id='call_99750276080216963')]
